# DeepSurv (PyCox) - Training Pipeline

Questo notebook definisce l'addestramento di un modello di Deep Learning per l'analisi di sopravvivenza (DeepSurv) utilizzando la libreria PyCox. A differenza dei modelli basati su alberi, le reti neurali richiedono una preparazione dei dati profondamente diversa e accorgimenti specifici per evitare l'overfitting.

---

### Fase 1: Caricamento e Preparazione Tensori
Il dataset viene caricato mantenendo la struttura di pulizia originale.
I valori codificati come `-1` vengono convertiti in `NaN` per essere processati correttamente.
Inoltre, per compatibilità con PyTorch e PyCox:
* Tutte le feature (`X`) vengono convertite esplicitamente nel formato numerico `float32`.
* Il target (`Y`) non è un array strutturato di scikit-survival, ma viene trasformato in una tupla di tensori `(tempi, eventi)`.

### Fase 2: Split Multiplo (Train / Validation / Test), Imputazione e Scaling
Il dataset subisce una doppia partizione:
1. **Test Set (25%):** Isolato per la valutazione finale.
2. **Validation Set (20% del Train residuo):** Creato appositamente per monitorare la rete durante l'addestramento e capire quando fermarsi.
* **Imputazione MICE:** Poiché le reti neurali e gli scaler non tollerano valori mancanti, applichiamo l'algoritmo MICE (`IterativeImputer`), addestrato esclusivamente sul Train, per risolvere i `NaN`.
* **Scaling Obbligatorio:** Le reti neurali sono estremamente sensibili alla scala delle variabili. Applichiamo uno `StandardScaler` (addestrato solo sul Train imputato) per standardizzare tutte le feature. L'imputer e lo scaler vengono poi salvati (`.pkl`) perché saranno fondamentali per processare correttamente i dati del test set.

### Fase 3: Architettura della Rete Neurale (DeepSurv)
Costruiamo un Multi-Layer Perceptron (MLPVanilla) configurato per limitare la complessità ed evitare di memorizzare il dataset (overfitting):



* **Strati Nascosti:** Due layer densi molto piccoli (32 e 16 nodi) proporzionati alle dimensioni ridotte del dataset.
* **Batch Normalization:** Attivata per stabilizzare e accelerare la convergenza dei gradienti durante l'addestramento.
* **Dropout (0.4):** Una tecnica di regolarizzazione aggressiva che "spegne" casualmente il 40% dei neuroni ad ogni iterazione, costringendo la rete ad apprendere pattern robusti e distribuiti.

### Fase 4: Addestramento ed Early Stopping
L'addestramento utilizza l'ottimizzatore Adam con un learning rate di 0.01 per un massimo di 200 epoche.
Per garantire la massima generalizzazione, introduciamo una callback di **Early Stopping** (patience=15): se l'errore sul Validation Set smette di diminuire per 15 epoche consecutive, l'addestramento si interrompe automaticamente, prevenendo l'overfitting.

### Fase 5: Salvataggio Modello
A differenza dei modelli classici di scikit-learn, la rete neurale viene salvata nel formato nativo di PyTorch (`.pt`), mentre lo scaler e l'imputer vengono salvati separatamente. Tutti e tre gli artefatti verranno ricaricati in fase di evaluation.

In [30]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
from pycox.models import CoxPH
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH    = "../data/dataset.csv"
MODEL_DIR    = "../models"
MODEL_PATH   = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH  = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25
VAL_SIZE     = 0.20

# ==============================================================================
# FASE 1: CARICAMENTO E VARIANCE THRESHOLD
# ==============================================================================
df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)

X_raw = df_clean.drop(columns=[TIME_COL, EVENT_COL])
min_positives = 15
binary_cols = [c for c in X_raw.columns if X_raw[c].nunique() == 2]
cols_to_drop = [c for c in binary_cols if X_raw[c].sum() < min_positives]

X_filtered = X_raw.drop(columns=cols_to_drop)
print(f"[!] Rimosse {len(cols_to_drop)} feature troppo sparse (< {min_positives} casi).")

# ==============================================================================
# FASE 2: MISSING INDICATORS E SPLIT
# ==============================================================================
# Creiamo indicatori per le colonne con molti NaN (Alcol e Pack Years)
cols_with_many_nan = ['exposures.alcohol_days_per_week',
                      'exposures.alcohol_drinks_per_day',
                      'exposures.pack_years_smoked']

for col in cols_with_many_nan:
    if col in X_filtered.columns:
        X_filtered[f"{col}_is_missing"] = X_filtered[col].isna().astype(float)

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_filtered, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

X_train, X_val, df_y_train, df_y_val = train_test_split(
    X_train_full, df_y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=df_y_train_full[EVENT_COL]
)

joblib.dump(X_train.columns.tolist(), FEATURES_PATH)

# ==============================================================================
# FASE 3: IMPUTAZIONE E SCALING
# ==============================================================================
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val_imp   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype('float32')
X_val_scaled   = scaler.transform(X_val_imp).astype('float32')

joblib.dump(imputer, IMPUTER_PATH)
joblib.dump(scaler, SCALER_PATH)

get_target = lambda df: (df[TIME_COL].values.astype('float32'), df[EVENT_COL].values.astype('float32'))
y_train = get_target(df_y_train)
y_val   = get_target(df_y_val)

# ==============================================================================
# FASE 4: TRAINING (DeepSurv)
# ==============================================================================
in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(in_features, [32, 16], 1, True, 0.4)
model = CoxPH(net, tt.optim.Adam(lr=0.01))

callbacks = [tt.callbacks.EarlyStopping(patience=15)]
log = model.fit(X_train_scaled, y_train, 32, 200, callbacks, val_data=(X_val_scaled, y_val), verbose=False)

model.save_net(MODEL_PATH)
print(f"Training completato. Feature finali: {in_features}")

[!] Rimosse 4 feature troppo sparse (< 15 casi).
Training completato. Feature finali: 36


# DeepSurv (PyCox) - Evaluation & Explainable AI

Questo notebook valuta le performance della rete neurale DeepSurv e ne esplora i meccanismi decisionali. Date le peculiarità architetturali di PyCox, la pipeline di valutazione e spiegabilità richiede alcuni passaggi di adattamento rispetto ai modelli standard.

---

### Fase 1: Ricostruzione Ambiente, Imputazione, Scaling e Baseline Hazards
Per valutare la rete, replichiamo rigorosamente l'ambiente di training:
* Ricarichiamo il dataset e ricreiamo il Test Set (25%) garantendo l'assenza di data leakage.
* Applichiamo il MICE (`IterativeImputer`) salvato durante il training per gestire i valori mancanti.
* Applichiamo lo `StandardScaler` salvato in addestramento per normalizzare il Test Set imputato.
* Ricostruiamo la medesima architettura di rete (nodi 32-16, dropout 0.4) e carichiamo i pesi dal file `.pt`.
* **Baseline Hazards:** A differenza di altri modelli, DeepSurv in PyCox calcola un rischio relativo. Per ottenere funzioni di sopravvivenza assolute ($S(t)$) e calcolare l'Integrated Brier Score, dobbiamo prima calcolare le "Baseline Hazards" passando l'intero set di training alla rete.

### Fase 2: Metriche Globali di Sopravvivenza
Le probabilità predette vengono appiattite e passate alle metriche standard:
* **C-index:** Misura l'accuratezza del ranking di rischio predetto dalla rete.
* **Integrated Brier Score (IBS):** Il dataframe di sopravvivenza predetto da PyCox viene interpolato per allinearlo ai step temporali di valutazione richiesti da `sksurv`, ottenendo così una misura di calibrazione dell'errore nel tempo.
* **Time-dependent AUC:** Valuta l'AUC a step temporali specifici (1, 3 e 5 anni).

### Fase 3: Stratificazione del Rischio (Validazione Clinica)
I pazienti del test set vengono separati in coorti ad "Alto Rischio" e "Basso Rischio" basandosi sulla mediana del rischio relativo (output grezzo della rete).
Successivamente, si mediano le curve di sopravvivenza (derivate dal dataframe di PyCox) per verificare visivamente l'efficacia del modello nel differenziare le prognosi.

### Fase 4: Explainable AI Globale (Permutation Feature Importance)
Le reti neurali sono notoriamente delle "black box". Per ispezionarle globalmente, avvolgiamo il modello DeepSurv in una classe wrapper (`PyCoxWrapper`) compatibile con scikit-learn. Questo permette di eseguire la Permutation Feature Importance, alterando una variabile scalata alla volta per misurare la conseguente diminuzione del C-index, rivelando così quali feature dominano l'apprendimento globale.

### Fase 5: Explainable AI Locale (SHAP) con Dati Imputati e Comprensibili
Per analizzare le decisioni sui singoli pazienti usiamo SHAP con un accorgimento tecnico cruciale:
Creiamo un wrapper personalizzato (`predict_risk_shap`) che accetta in input i dati **imputati ma non scalati**, applica lo scaler internamente e restituisce il rischio.
Questo permette al KernelExplainer di SHAP di campionare e restituire grafici (Summary Plot e Waterfall Plot) che mostrano i **valori originali e comprensibili delle feature** (es. "Età = 65"), anziché valori astratti standardizzati (es. "Età = 1.24"), rendendo l'interpretazione clinicamente utile e intellegibile.

In [31]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
import shap
import matplotlib.pyplot as plt
from pycox.models import CoxPH
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                            integrated_brier_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR   = "results"
DATA_PATH     = "../data/dataset.csv"
MODEL_DIR     = "../models"
MODEL_PATH    = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH   = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH  = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f in [MODEL_PATH, SCALER_PATH, IMPUTER_PATH, FEATURES_PATH]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"File {_f} non trovato. Esegui prima il training.")

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25

# ==============================================================================
# 1. RICOSTRUZIONE AMBIENTE E PREPROCESSING
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - DEEPSURV")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)
feature_cols_final = joblib.load(FEATURES_PATH)

cols_indicators = ['exposures.alcohol_days_per_week',
                   'exposures.alcohol_drinks_per_day',
                   'exposures.pack_years_smoked']

X_proc = df_clean.drop(columns=[TIME_COL, EVENT_COL])
for col in cols_indicators:
    if col in X_proc.columns:
        X_proc[f"{col}_is_missing"] = X_proc[col].isna().astype(float)

X_proc = X_proc[feature_cols_final]

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_proc, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

y_train_full_sksurv = Surv.from_arrays(event=df_y_train_full[EVENT_COL].astype(bool), time=df_y_train_full[TIME_COL])
y_test_sksurv = Surv.from_arrays(event=df_y_test[EVENT_COL].astype(bool), time=df_y_test[TIME_COL])

imputer = joblib.load(IMPUTER_PATH)
scaler  = joblib.load(SCALER_PATH)

X_train_full_imp = pd.DataFrame(imputer.transform(X_train_full), columns=X_train_full.columns)
X_test_imp       = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

X_train_full_scaled = scaler.transform(X_train_full_imp).astype('float32')
X_test_scaled       = scaler.transform(X_test_imp).astype('float32')

in_features = X_train_full_scaled.shape[1]
net = tt.practical.MLPVanilla(in_features, [32, 16], 1, True, 0.4)
model = CoxPH(net, tt.optim.Adam())
model.load_net(MODEL_PATH, weights_only=False)

y_train_pycox = (df_y_train_full[TIME_COL].values.astype('float32'), df_y_train_full[EVENT_COL].values.astype('float32'))
_ = model.compute_baseline_hazards(X_train_full_scaled, y_train_pycox)

risk_test = model.predict(X_test_scaled).flatten()
surv_df = model.predict_surv_df(X_test_scaled)

print(f"[+] Ambiente ricostruito: {in_features} feature attive.")

# ==============================================================================
# FASE 1: METRICHE GLOBALI
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 1: METRICHE GLOBALI")
print("=" * 62)

c_index = concordance_index_censored(y_test_sksurv['event'], y_test_sksurv['time'], risk_test)[0]

lb = np.percentile(y_test_sksurv['time'], 10)
ub = np.percentile(y_test_sksurv['time'], 90)
ibs_times = np.linspace(lb, ub, 20)

preds_interp = np.array([np.interp(ibs_times, surv_df.index.values, surv_df.iloc[:, i].values) for i in range(surv_df.shape[1])])
ibs = integrated_brier_score(y_train_full_sksurv, y_test_sksurv, preds_interp, ibs_times)

print(f"  C-index (test set) : {c_index:.4f}")
print(f"  IBS                : {ibs:.4f}")

# ==============================================================================
# FASE 2: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 2: STRATIFICAZIONE CLINICA")
print("=" * 62)

median_risk = np.median(risk_test)
mask_high, mask_low = risk_test > median_risk, risk_test <= median_risk

plt.figure(figsize=(10, 6))
plt.step(surv_df.index, surv_df.loc[:, mask_low].mean(axis=1), where="post", label="Basso Rischio", color="#2ca02c")
plt.step(surv_df.index, surv_df.loc[:, mask_high].mean(axis=1), where="post", label="Alto Rischio", color="#d62728")
plt.title("Sopravvivenza Media: Alto vs Basso Rischio (DeepSurv)")
plt.ylabel("S(t)")
plt.xlabel("Giorni")
plt.legend()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_survival_curves.png"))
plt.close()

# ==============================================================================
# FASE 3: SHAP (Interpretazione con Indicatori)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: XAI LOCALE (SHAP)")
print("=" * 62)

def predict_risk_shap(X_imp):
    X_scaled = scaler.transform(X_imp).astype('float32')
    return model.predict(X_scaled).flatten()

background = shap.sample(X_train_full_imp, 100, random_state=RANDOM_STATE)
explainer = shap.Explainer(predict_risk_shap, background)

print("  Calcolo SHAP (BeeSwarm)...")
shap_values = explainer(X_test_imp)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Summary: Impatto sul Rischio (con Missing Indicators)")
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_shap_summary.png"), bbox_inches="tight")
plt.close()

idx_max = int(np.argmax(risk_test))
plt.figure(figsize=(12, 6))
shap.plots.waterfall(shap_values[idx_max], show=False)
plt.title(f"SHAP Waterfall: Paziente Alto Rischio (ID Test {idx_max})")
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_waterfall_high.png"), bbox_inches="tight")
plt.close()

print(f"  [+] Tutti i grafici salvati in: {RESULTS_DIR}")

  EVALUATION & XAI - DEEPSURV
[+] Ambiente ricostruito: 36 feature attive.

  FASE 1: METRICHE GLOBALI
  C-index (test set) : 0.7170
  IBS                : 0.1861

  FASE 2: STRATIFICAZIONE CLINICA

  FASE 3: XAI LOCALE (SHAP)
  Calcolo SHAP (BeeSwarm)...
  [+] Tutti i grafici salvati in: results
